Install Required Libraries

In [1]:
# Cell 1: Install Required Libraries
!pip uninstall -y torchaudio torchvision
!pip install -q --upgrade transformers accelerate

Environment & Hardware Setup

In [2]:
# Cell 2: Environment & Hardware Setup
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Environment Ready! Operating on device: {device}")

✅ Environment Ready! Operating on device: cuda


Load Model and Tokenizer

In [3]:
# Cell 3: Load Model and Tokenizer
model_id = "Qwen/Qwen1.5-0.5B-Chat"

# Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.eval()

print("✅ Qwen Chat Model & Tokenizer loaded successfully!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Qwen Chat Model & Tokenizer loaded successfully!


Memory Manager & Prompt Context Construction

In [4]:
# Cell 4: Memory Manager using Chat Template

class ConversationMemory:
    """Class to manage chat history and generate prompt context using standard Chat Templates."""
    def __init__(self, system_prompt="You are a helpful and context-aware AI assistant."):
        self.system_prompt = system_prompt
        self.history = [{"role": "system", "content": self.system_prompt}]

    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})

    def add_ai_message(self, message):
        self.history.append({"role": "assistant", "content": message})

    def build_context_prompt(self, tokenizer):
        # Apply standard HuggingFace chat template directly to conversation history
        return tokenizer.apply_chat_template(
            self.history,
            tokenize=False,
            add_generation_prompt=True
        )

    def clear(self):
        self.history = [{"role": "system", "content": self.system_prompt}]

# Initialize memory instance
memory = ConversationMemory()
print("✅ Conversation Memory Manager Initialized!")

✅ Conversation Memory Manager Initialized!


Chat Assistant Loop & Verification Test

In [5]:
# Cell 5: Response Generation & Memory Verification Test

def generate_chat_response(memory_instance):
    prompt_text = memory_instance.build_context_prompt(tokenizer)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.2,       # Low temperature to prevent hallucinations
            top_p=0.9,
            repetition_penalty=1.1 # Penalty to prevent repetitive output
        )

    # Extract response portion only
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# --- 1. Clear previous history to avoid prompt pollution ---
memory.clear()

# --- 2. Run automated verification test ---
print("🚀 --- RUNNING AUTOMATED MEMORY & CONTEXT TEST --- 🚀\n")

# Step 1: User introduces info
user_msg_1 = "My name is Mazen and I am 19 years old and studying Computer Science."
print(f"👤 User: {user_msg_1}")
memory.add_user_message(user_msg_1)
ai_resp_1 = generate_chat_response(memory)
memory.add_ai_message(ai_resp_1)
print(f"🤖 AI: {ai_resp_1}\n")

# Step 2: User asks to recall info
user_msg_2 = "What is my name, how old am I, and what am I studying?"
print(f"👤 User: {user_msg_2}")
memory.add_user_message(user_msg_2)
ai_resp_2 = generate_chat_response(memory)
memory.add_ai_message(ai_resp_2)
print(f"🤖 AI: {ai_resp_2}\n")

print("=========================================")
print("✅ CONTEXT-AWARE MEMORY TEST PASSED SUCCESSFULLY!")
print("=========================================")

🚀 --- RUNNING AUTOMATED MEMORY & CONTEXT TEST --- 🚀

👤 User: My name is Mazen and I am 19 years old and studying Computer Science.
🤖 AI: Hello Mazen! It's great to meet you. What can I assist you with today?

👤 User: What is my name, how old am I, and what am I studying?
🤖 AI: Your name is Mazen. You are currently 19 years old and you are studying Computer Science.

✅ CONTEXT-AWARE MEMORY TEST PASSED SUCCESSFULLY!
